# DL dataset choices and ablation studies

This notebook addresses two core rubric items on NASA C-MAPSS FD001:
1. **DL Dataset Choices and Justification:**
   - Evaluates sequence window length coverage ($w \in \{15, 30, 50\}$) on training cycles and test engines.
   - Quantifies padding requirements (shortest test engine is 31 cycles).
   - Justifies the piece-wise linear target RUL cap at 125 cycles against raw uncapped RUL.
   - Verifies sequence operating regime normalisation against the single-regime property of FD001.
2. **Regularization and Ablation Studies:**
   - Single-factor-at-a-time ablations systematically removing or altering one component from the chosen sequence model (Transformer encoder, $w=30$, cap 125, dropout 0.1, weight decay $10^{-4}$, patience 10 on 20% nested validation, 14 useful sensors).
   - Evaluates window lengths ($w=15, 30, 50$), target capping, regularisation components (dropout, weight decay, early stopping), input sensors (all 14 vs. top SHAP sensors), and hybrid architectural branches.
   - Scored on the standard 5-fold engine-grouped cross-validation without test set leakage.


In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rul.ablations import cap_distribution, dataset_coverage, regime_variation

RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
ABLATIONS_CSV = RESULTS / "ablations.csv"

LOAD_SAVED = True  # Set to False to re-run all 5-fold cross-validations from scratch (~15-20 min on CPU)
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", None)

print("Environment initialized.")
print(f"LOAD_SAVED = {LOAD_SAVED}")


Environment initialized.
LOAD_SAVED = True


## 1. Deep Learning Dataset Choices and Justification

### Window Length Coverage and Padding Trade-off
Sequence models map a temporal history of $w$ cycles of 14 sensor readings to remaining useful life (RUL).
Because an engine has only $t$ observations at cycle $t$, windows for cycles $t < w$ have fewer than $w$ historical cycles.
Rather than discard healthy early cycles, we left-pad each window by repeating that engine's initial cycle reading.
Since healthy sensors are nearly stationary, repeating cycle 1 introduces no false degradation slope and strictly avoids borrowing readings across engines.

However, window length introduces a direct physical and operational trade-off:
- Shortest training engine has 128 cycles.
- **Shortest test engine (unit 1) has only 31 cycles.**


In [2]:
cov_df = dataset_coverage((15, 30, 50))
cov_df[[
    "window", "train_cycles_total", "train_cycles_kept", "train_kept_pct",
    "train_windows_padded", "test_engines_total", "test_engines_kept",
    "test_engines_padded", "shortest_test_engine"
]]


,window,train_cycles_total,train_cycles_kept,train_kept_pct,train_windows_padded,test_engines_total,test_engines_kept,test_engines_padded,shortest_test_engine
0,15,20631,19231,93.214,1400,100,100,0,31
1,30,20631,17731,85.943,2900,100,100,0,31
2,50,20631,15731,76.249,4900,100,93,7,31


### Target Capping and Operating Regime Equivalence
1. **Target RUL Cap at 125 cycles:** Before degradation begins, turbofan sensors exhibit stationary baselines identical between an engine with 300 cycles remaining and one with 150 cycles remaining. Forcing networks to regress uncapped RUL over healthy cycles penalizes them for unobservable variance and pulls gradients away from late wear. Clipping RUL at 125 cycles establishes a healthy plateau for 38.9% of cycles.
2. **Operating Regime Normalisation:** FD001 features constant sea-level operating settings with standard deviations $\le 0.0022$. Standardizing sensors by operating regime is mathematically identical to global standardization across training engines.


In [3]:
cap_stats = cap_distribution(125)
print("RUL Target Distribution (Cap = 125):")
for k, v in cap_stats.items():
    print(f"  {k:15s}: {v:10.2f}")

print("\nOperating Setting Variability (FD001):")
display(regime_variation())


RUL Target Distribution (Cap = 125):
  raw_mean       :     107.81
  raw_std        :      68.88
  raw_median     :     103.00
  raw_max        :     361.00
  capped_mean    :      86.83
  capped_std     :      41.67
  capped_median  :     103.00
  capped_max     :     125.00
  n_capped       :    8031.00
  pct_capped     :      38.93

Operating Setting Variability (FD001):


,setting,min,max,mean,std,n_unique
0,setting_1,-8.700e-03,8.700e-03,-8.870e-06,2.187e-03,158
1,setting_2,-6.000e-04,6.000e-04,2.351e-06,2.931e-04,13
2,setting_3,1.000e+02,1.000e+02,1.000e+02,0.000e+00,1


### Visualising Dataset Coverage and Target Distribution
![Dataset Coverage and Padding Trade-off](../figures/ablation_coverage_padding.png)


In [4]:
from scripts.plot_coverage import plot_dataset_diagnostics
 
plot_dataset_diagnostics()


Saved: figures/ablation_coverage_padding.png


## 2. Regularization and Ablation Studies

### Experimental Methodology
Following strict scientific ablation discipline:
1. **Single Factor at a Time:** Every ablation modifies *exactly one* component relative to the baseline Transformer sequence model ($w=30$, cap 125, dropout 0.1, weight decay $10^{-4}$, patience 10 on 20% nested validation, 14 useful sensors).
2. **Identical 5-Fold Engine Cross-Validation:** All runs use the shared engine folds from Phase 1, scoring every cycle against RUL capped at 125.
3. **Two runs, each with its own baseline:** the same baseline configuration gives CV RMSE 13.00 with default PyTorch threads and 13.17 with one thread (13.28 in `dl_cv.csv`), because the order of floating point operations changes training. The rows come from two runs, so each is compared only with the baseline of its own run (`run` and `reference` columns, `rmse_delta`).
4. **Evidence criterion:** a difference counts only if it is larger than the fold standard deviation of both rows and than the run to run variation of about 0.3 cycles.
5. **No early stopping** means 40 epochs keeping the last epoch's weights (`fit_net(..., restore_best=False)`); otherwise `fit_net` restores the best early stopping epoch.


In [5]:
if LOAD_SAVED and ABLATIONS_CSV.exists():
    ablations_df = pd.read_csv(ABLATIONS_CSV)
    print(f"Loaded {len(ablations_df)} ablations from {ABLATIONS_CSV.relative_to(ROOT)}")
else:
    print("Running full ablation suite via scripts/run_ablations.py...")
    from scripts.run_ablations import main as run_all_ablations
    run_all_ablations()
    ablations_df = pd.read_csv(ABLATIONS_CSV)

display(ablations_df[[
    "ablation", "changed_factor", "run", "rmse_mean", "rmse_std", "rmse_delta",
    "mae_mean", "rmse_late_mean", "takeaway"
]])


Loaded 15 ablations from results/ablations.csv


,ablation,changed_factor,run,rmse_mean,rmse_std,rmse_delta,mae_mean,rmse_late_mean,takeaway
0,Baseline (Transformer),"None (reference: w=30, cap 125, dropout 0.1, weight decay 1e-4, early stopping, 14 sensors)",default torch threads,13.003,0.674,0.000,9.302,7.849,Reference for the rows run with default threads.
1,Window length: w=15,window = 15 (reduced from 30),default torch threads,16.960,0.484,3.958,12.355,14.178,Much worse overall and near failure: 15 cycles of history is too little.
2,Window length: w=50,window = 50 (increased from 30),default torch threads,11.343,0.523,-1.660,8.062,6.424,"Clearly better on CV, on every fold, also on cycles >= 50 where no window is padded; not adopted because the test set was already opened (#48)."
3,Regularisation: No dropout,dropout = 0.0 (down from 0.1),default torch threads,13.545,0.839,0.542,9.734,8.139,Within the fold spread: no measurable effect.
4,Regularisation: No weight decay,weight_decay = 0.0 (down from 1e-4),default torch threads,13.467,0.712,0.464,9.524,7.687,Within the fold spread: no measurable effect.
5,Inputs: Top 3 SHAP sensors,"sensors = [s_4, s_15, s_2] (down from 14)",default torch threads,18.701,1.499,5.698,13.254,15.482,Much worse overall and near failure: three sensors are not enough.
6,Inputs: Top 5 SHAP sensors,"sensors = [s_4, s_15, s_2, s_14, s_11] (down from 14)",default torch threads,15.444,1.123,2.441,10.665,11.637,Recovers part of the gap from top 3 but stays well behind all 14 sensors.
7,Hybrid: Sequence branch only,HybridNet with 0 engineered features (Sequence + MLP head),default torch threads,13.336,0.337,0.333,9.427,7.341,Matches the plain Transformer within the fold spread; lower late RMSE than the two-branch hybrid.
8,Hybrid: Feature branch only,HybridNet with no sequence window (142 features + MLP head),default torch threads,15.288,0.910,2.286,11.319,12.332,Far behind the Transformer and LSTM: the sensor window carries most of the signal.
9,Hybrid: Two-Branch (Seq + Feat),Both branches combined (30-cycle window + 142 features),default torch threads,13.471,0.532,0.468,9.767,7.532,"No better than the sequence branch alone, overall or near failure."


### Is the w=50 gain only padding?
A padded window shows how early in life an engine is. Scoring only cycles >= 50, where no window of either length is padded, separates real history from that cue.


In [6]:
window_check = pd.read_csv(RESULTS / "ablation_window_unpadded.csv")
display(window_check.pivot(index="fold", columns="window", values="rmse_cycles_ge_50"))
window_check.groupby("window")[["rmse_all_cycles", "rmse_cycles_ge_50"]].agg(["mean", "std"])


window,30,50
fold,,
0,14.574,11.878
1,12.889,11.848
2,14.125,12.879
3,14.517,11.238
4,14.856,11.977


rmse_all_cycles        rmse_cycles_ge_50       
                  mean    std              mean    std
window                                                
30              13.171  0.717            14.192  0.774
50              11.186  0.336            11.964  0.589

### Visualising Ablation Performance vs. Baseline
Each bar is a row's CV RMSE minus the baseline of its own run; the error bar is that row's fold standard deviation.


In [7]:
from scripts.plot_ablations import plot_ablations_comparison

plot_ablations_comparison()


Saved: figures/ablation_cv_comparison.png


## 3. Findings

1. **Window length.** $w=15$ is clearly worse ($+3.96$). $w=50$ is clearly better ($-1.66$, 2.5 to 3 fold standard deviations), and stays better on cycles $\ge 50$ where nothing is padded (11.96 vs 14.19, every fold), so longer history genuinely helps on CV. We keep $w=30$ only because the window was fixed before the test set was opened in #48; switching now would be a choice made after seeing test results. It is reported as a limitation.
2. **Target cap.** Without the cap, even with predictions clipped at 125, RMSE is 2.25 worse and the fold spread doubles. On its own uncapped target the RMSE is 38.57.
3. **Regularisation.** No dropout ($+0.54$) and no weight decay ($+0.46$) are within the fold spread: no measurable effect. No early stopping is clearly worse ($+2.48$), and all three off ($+2.83$) is about the same, so early stopping is the regulariser that matters.
4. **Inputs.** Top 3 SHAP sensors: $+5.70$. Top 5: $+2.44$, still well behind all 14.
5. **Hybrid branches.** Sequence branch alone $\approx$ plain Transformer ($+0.33$). Feature branch alone $+2.29$. Both branches are no better than the sequence branch alone, overall or near failure (late 7.53 vs 7.34).

### Viva: you must be able to explain
- **Why an ablation changes one factor at a time:** with two changes you cannot tell which one caused the difference.
- **Why a gap smaller than the fold spread is not evidence:** a different split of engines into folds could move it by that much.
- **Why each row needs a baseline from its own run:** the thread count alone moves the baseline from 13.00 to 13.17.
- **Why window length trades history against padding:** longer windows see more history but pad more early cycles, and at $w=50$ seven test engines are shorter than the window.
